# PyCaret Redux — Complete Classification Demo

This notebook demonstrates **every feature** of the pycaret_redux classification library.

## 1. Setup & Data Profiling

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer(as_frame=True)
df = data.frame
df["target"] = data.target

print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")
df.head()

In [ ]:
from pycaret_redux import ClassificationExperiment

exp = ClassificationExperiment()
exp.setup(
    data=df,
    target="target",
    train_size=0.8,
    normalize=True,
    normalize_method="zscore",
    session_id=42,
    fold=5,
    profile=True,  # Shows class distribution, missing values, correlations
)

## 2. Inspect State

In [ ]:
# List available models (including HistGradientBoosting)
exp.models()

In [ ]:
# List available metrics
exp.get_metrics()

In [ ]:
# Inspect experiment config
print("Seed:", exp.get_config("seed"))
print("Target:", exp.get_config("target_name"))
print("Multiclass:", exp.get_config("is_multiclass"))
print("Feature types:", {k: len(v) for k, v in exp.get_config("feature_types").items() if v})

In [ ]:
# Inspect the preprocessing pipeline
# This dataset is all-numeric, so only the numeric branch appears.
# With mixed data you'd also see categorical (SmartEncoder) and other branches.
exp.get_pipeline()

## 3. Compare Models

In [ ]:
best = exp.compare_models(
    include=["lr", "dt", "rf", "nb", "knn", "ada", "gbc", "hgbc", "et", "lda", "ridge"],
    sort="Accuracy",
    n_select=1,
)

In [ ]:
# pull() returns the last result as a DataFrame
exp.pull()

## 4. Create Models

In [ ]:
lr = exp.create_model("lr")

In [ ]:
rf = exp.create_model("rf")

In [ ]:
# New: HistGradientBoosting (sklearn's built-in LightGBM equivalent)
hgbc = exp.create_model("hgbc")

## 5. Tune Models

### Standard RandomizedSearchCV

In [ ]:
tuned_rf = exp.tune_model(rf, n_iter=20, optimize="F1")

### HalvingRandomSearchCV (successive halving — much faster)

In [ ]:
tuned_hgbc = exp.tune_model(hgbc, search_library="halving", optimize="Accuracy")

## 6. Ensembles

In [ ]:
# Soft voting
blended = exp.blend_models([lr, tuned_rf], method="soft")

In [ ]:
# Stacking
stacked = exp.stack_models([lr, tuned_rf])

In [ ]:
# Bagging
bagged = exp.ensemble_model(tuned_rf, n_estimators=10)

## 7. AutoML

One-liner: compare → tune top N → ensemble.

In [ ]:
automl_model = exp.automl(
    optimize="Accuracy",
    n_top=3,
    tune_n_iter=10,
    ensemble="blend",
)

## 8. Plots

In [ ]:
%matplotlib inline

In [ ]:
exp.plot_model(tuned_rf, plot="auc");

In [ ]:
exp.plot_model(tuned_rf, plot="confusion_matrix");

In [ ]:
exp.plot_model(tuned_rf, plot="pr");

In [ ]:
exp.plot_model(tuned_rf, plot="feature");

In [ ]:
# NEW: Permutation importance (model-agnostic, with error bars)
exp.plot_model(tuned_rf, plot="permutation");

In [ ]:
exp.plot_model(tuned_rf, plot="class_report");

In [ ]:
exp.plot_model(tuned_rf, plot="threshold");

In [ ]:
exp.plot_model(tuned_rf, plot="calibration");

In [ ]:
exp.plot_model(tuned_rf, plot="lift");

In [ ]:
exp.plot_model(tuned_rf, plot="gain");

In [ ]:
exp.plot_model(tuned_rf, plot="ks");

### Learning Curve

In [ ]:
exp.plot_model(tuned_rf, plot="learning");

### Validation Curve

Shows train/validation score vs a hyperparameter value — find the sweet spot.

In [ ]:
# Validation curve for RF (auto-detects n_estimators parameter)
exp.plot_model(tuned_rf, plot="vc");

## 10. Calibration & Threshold Optimization

`calibrate_model` now shows **before/after Brier Score and Log Loss**.

In [ ]:
exp.evaluate_model(tuned_rf)

In [ ]:
preds = exp.predict_model(tuned_rf, verbose=False)
preds.head(10)

In [ ]:
# Raw probability scores
preds_raw = exp.predict_model(tuned_rf, raw_score=True, verbose=False)
preds_raw[["prediction_label", "prediction_score_0", "prediction_score_1"]].head(10)

## 10. Calibration & Threshold Optimization

In [ ]:
calibrated = exp.calibrate_model(tuned_rf, method="sigmoid")

In [ ]:
model, best_threshold = exp.optimize_threshold(tuned_rf, optimize="F1")

# Predict with optimized threshold
preds_opt = exp.predict_model(tuned_rf, probability_threshold=best_threshold, verbose=False)
preds_opt["prediction_label"].value_counts()

## 11. Statistical Model Comparison

### McNemar's test (pairwise, on predictions)

In [ ]:
dt = exp.create_model("dt", verbose=False)
exp.compare_model_stats(tuned_rf, dt, test="mcnemar")

### Wilcoxon signed-rank test (pairwise, on CV folds)

In [ ]:
exp.compare_model_stats(tuned_rf, lr, metric="Accuracy", test="wilcoxon")

### Cochran's Q test (3+ classifiers)

In [ ]:
exp.compare_multiple_stats([lr, tuned_rf, dt, hgbc])

### 5x2cv paired F-test (Dietterich's method)

In [ ]:
exp.compare_5x2cv(lr, tuned_rf)

## 12. Nested Cross-Validation

Unbiased evaluation: inner loop tunes hyperparameters, outer loop estimates performance.

In [ ]:
nested_scores = exp.nested_cv("rf", n_iter=10)

## 13. Bias-Variance Diagnostic

In [ ]:
exp.diagnose_bias_variance(tuned_rf)

In [ ]:
# Compare with a simpler model
exp.diagnose_bias_variance(dt)

## 14. OOB Evaluation

Free validation for forest/bagging models using out-of-bag samples.

In [ ]:
oob = exp.get_oob_score(tuned_rf)
print(f"OOB Accuracy: {oob}")

## 15. Data Drift Detection

In [ ]:
# Check drift between training data and test data
drift_report = exp.check_drift(exp.X_test)
drift_report

## 16. Custom Metrics

In [ ]:
from sklearn.metrics import balanced_accuracy_score

exp.add_metric(id="bal_acc", name="Balanced Accuracy", score_func=balanced_accuracy_score)
gbc = exp.create_model("gbc")

In [ ]:
exp.remove_metric("bal_acc")
print("Custom metric removed.")

## 17. Finalize, Save & Load

In [ ]:
final_model = exp.finalize_model(tuned_rf)
exp.save_model(final_model, "breast_cancer_rf")

In [ ]:
loaded = exp.load_model("breast_cancer_rf")
print(f"Loaded model: {type(loaded).__name__}")

In [ ]:
# Standalone deployment: load artifact with pipeline
from pycaret_redux.persistence.serialization import load_model, predict_from_artifact

artifact = load_model("breast_cancer_rf")
print(f"Pipeline bundled: {artifact.pipeline is not None}")
print(f"Target: {artifact.target_name}")

# Predict from raw data (pipeline handles preprocessing)
standalone_preds = predict_from_artifact(artifact, exp.X_test)
print(f"Predictions: {standalone_preds[:5]}")

## 19. Multiclass Classification

## 18. Mixed Data with Categorical Features & Labels

Demonstrates categorical encoding (SmartEncoder), feature labels for readable plots, and the `drop_first_ohe` option.

In [ ]:
import numpy as np

# Create a mixed dataset with numeric codes for categorical features
np.random.seed(42)
n = 500
mixed_df = pd.DataFrame({
    "age": np.random.randint(18, 80, n),
    "income": np.random.normal(50000, 15000, n).round(0),
    "bmi": np.random.normal(25, 5, n).round(1),
    "smoking": np.random.choice([0, 1], n),                          # binary
    "exercise": np.random.choice([0, 1, 2], n),                      # ordinal
    "marital_status": np.random.choice([1, 2, 3], n),                # nominal
    "education": np.random.choice([1, 2, 3, 4], n),                  # ordinal
    "target": np.random.choice([0, 1], n, p=[0.7, 0.3]),             # imbalanced
})

# Define labels for numeric codes — these will appear in plots
feature_labels = {
    "smoking": {0: "No", 1: "Yes"},
    "exercise": {0: "Never", 1: "Sometimes", 2: "Regular"},
    "marital_status": {1: "Married", 2: "Single", 3: "Divorced"},
    "education": {1: "High School", 2: "Bachelor", 3: "Master", 4: "PhD"},
}
target_labels = {0: "Healthy", 1: "At Risk"}

print("Dataset shape:", mixed_df.shape)
print("\nFeature types:")
print(mixed_df.dtypes)
print("\nSample rows:")
mixed_df.head()

In [ ]:
# Setup with categorical features specified and labels for plots
exp_mixed = ClassificationExperiment()
exp_mixed.setup(
    data=mixed_df,
    target="target",
    # Tell the library which features are categorical (they're stored as ints)
    categorical_features=["smoking", "exercise", "marital_status", "education"],
    # Labels for readable plots
    feature_labels=feature_labels,
    target_labels=target_labels,
    # drop_first avoids dummy variable trap for logistic regression
    drop_first_ohe=True,
    # Handle the 70/30 class imbalance
    fix_imbalance=True,
    fix_imbalance_method="class_weight",
    normalize=True,
    session_id=42,
    fold=5,
    profile=True,
)

In [ ]:
# Pipeline now shows both numeric and categorical branches
exp_mixed.get_pipeline()

## 20. Cleanup

In [ ]:
# Feature importance with labels — shows "smoking: Yes" instead of "smoking_1"
exp_mixed.plot_model(best_mixed, plot="feature");

In [ ]:
# Permutation importance with labels
exp_mixed.plot_model(best_mixed, plot="permutation");

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True).frame

exp_multi = ClassificationExperiment()
exp_multi.setup(data=iris, target="target", session_id=123, fold=5)

In [ ]:
best_multi = exp_multi.compare_models(include=["lr", "dt", "rf", "nb", "knn", "hgbc"])

In [ ]:
exp_multi.plot_model(best_multi, plot="confusion_matrix");

In [ ]:
preds_multi = exp_multi.predict_model(best_multi, verbose=False)
preds_multi[["prediction_label", "prediction_score"]].head(10)

## 19. Cleanup

In [ ]:
import os

if os.path.exists("breast_cancer_rf.joblib"):
    os.remove("breast_cancer_rf.joblib")
    print("Cleaned up saved model file.")

print("\nAll features tested successfully!")